<a href="https://colab.research.google.com/github/nawresssjebali/building_directory/blob/main/Scrape_Bwy.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q selenium
!apt-get update -qq
!apt-get install -qq chromium-browser chromium-chromedriver

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Preconfiguring packages ...
Selecting previously unselected package chromium-browser.
(Reading database ... 119113 files and directories currently installed.)
Preparing to unpack .../chromium-browser_1%3a85.0.4183.83-0ubuntu2.22.04.1_amd64.deb ...
=> Installing the chromium snap
==> Checking connectivity with the snap store
===> System doesn't have a working snapd, skipping
Unpacking chromium-browser (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Selecting previously unselected package chromium-chromedriver.
Preparing to unpack .../chromium-chromedriver_1%3a85.0.4183.83-0ubuntu2.22.04.1_amd64.deb ...
Unpacking chromium-chromedriver (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Setting up chromium-browser (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Setting up chromium-chromedriver (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Proce

In [ ]:
!which chromium-browser chromedriver
!chromium-browser --version
!chromedriver --version

/usr/bin/chromium-browser
/usr/bin/chromedriver

Command '/usr/bin/chromium-browser' requires the chromium snap to be installed.
Please install it with:

snap install chromium


Command '/usr/bin/chromedriver' requires the chromium snap to be installed.
Please install it with:

snap install chromium



In [ ]:
!pip install -q -U selenium
!apt-get purge -qq chromium-browser chromium-chromedriver -y
!wget -q -O /tmp/chrome.deb https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get install -qq /tmp/chrome.deb
!google-chrome --version

(Reading database ... 119134 files and directories currently installed.)
Removing chromium-chromedriver (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Removing chromium-browser (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Processing triggers for mailcap (3.70+nmu1ubuntu1) ...
Processing triggers for hicolor-icon-theme (0.17-2) ...
(Reading database ... 119113 files and directories currently installed.)
Purging configuration files for chromium-browser (1:85.0.4183.83-0ubuntu2.22.04.1) ...
Google Chrome 151.0.7922.71 


In [ ]:
import argparse
import csv
import os
import re
import time

from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.common.exceptions import TimeoutException, WebDriverException
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait

BASE_URL = "https://portal.bwy.org.uk/user/public-entries/{}"

READY_MARKER = "Teacher:"
NOT_FOUND_MARKERS = ("page not found", "not found", "404")


def make_driver():
    opts = Options()
    opts.add_argument("--headless=new")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--window-size=1280,2400")
    opts.add_argument(
        "user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/124.0 Safari/537.36"
    )
    opts.binary_location = "/usr/bin/google-chrome"
    # No Service() pin — Selenium Manager downloads a matching chromedriver
    # for this exact Chrome version automatically.
    return webdriver.Chrome(options=opts)


def load_rendered_html(driver, url, timeout=12):
    driver.get(url)
    try:
        WebDriverWait(driver, timeout).until(
            lambda d: READY_MARKER in d.page_source
            or any(m in d.page_source.lower() for m in NOT_FOUND_MARKERS)
        )
    except TimeoutException:
        pass
    return driver.page_source

FIELDNAMES = [
    "id", "title", "teacher_name", "address", "schedule_next_session",
    "schedule_recurrence", "style_tags", "qualification_note", "description",
    "how_to_book", "equipment_needed", "teacher_bio", "price", "suitable_for",
    "safeguarding", "image_url", "url",
]


def clean(text):
    if text is None:
        return ""
    return re.sub(r"\s+", " ", text).strip()


def get_section_text(soup, heading_text):
    heading = soup.find(
        lambda tag: tag.name in ("h5", "h6", "strong", "b", "p")
        and heading_text.lower() in tag.get_text(strip=True).lower()
    )
    if not heading:
        return ""
    nxt = heading.find_next_sibling()
    if nxt:
        return clean(nxt.get_text(" ", strip=True))
    parent_next = heading.parent.find_next_sibling() if heading.parent else None
    if parent_next:
        return clean(parent_next.get_text(" ", strip=True))
    return ""


def parse_entry(entry_id, html):
    soup = BeautifulSoup(html, "html.parser")

    title = ""
    if soup.title and soup.title.string:
        title = clean(soup.title.string)
        title = re.sub(r"\s*\|\s*The British Wheel of Yoga\s*$", "", title, flags=re.I)
    if not title:
        heading = soup.find(["h1", "h2", "h3", "h4"])
        title = clean(heading.get_text()) if heading else ""

    lowered = html.lower()
    if (
        not title
        or "teacher:" not in lowered
        or any(m in lowered for m in ("page not found", "404 error"))
    ):
        return None

    full_text = soup.get_text("\n", strip=True)

    teacher_match = re.search(r"Teacher:\s*(.+)", full_text)
    teacher_name = clean(teacher_match.group(1).splitlines()[0]) if teacher_match else ""

    address = ""
    addr_match = re.search(
        r"\n([^\n]{5,120}?,\s*[^\n]{2,60}?,\s*[A-Z]{1,2}\d[A-Z\d]?\s*\d[A-Z]{2})\n",
        full_text,
    )
    if addr_match:
        address = clean(addr_match.group(1))

    next_session = ""
    ns_match = re.search(r"Next Session:\s*([\d/]+)", full_text)
    if ns_match:
        next_session = ns_match.group(1)

    recurrence = ""
    rec_match = re.search(r"(Occurs [^\n]+)", full_text)
    if rec_match:
        recurrence = clean(rec_match.group(1))

    style_tags = ""
    style_match = re.search(
        r"(?:" + re.escape(recurrence) + r"|" + re.escape(next_session) + r")\s*\n([^\n]+)\n\s*Teacher:",
        full_text,
    )
    if style_match:
        style_tags = clean(style_match.group(1))

    qualification_note = get_section_text(soup, "This entry is made by teacher")
    if not qualification_note:
        qual_match = re.search(r"(This entry is made by teacher[^\n]*(?:\n[^\n]*){0,4})", full_text)
        qualification_note = clean(qual_match.group(1)) if qual_match else ""

    description = get_section_text(soup, "Entry Description")
    how_to_book = get_section_text(soup, "How to Book")
    equipment = get_section_text(soup, "Equipment Needed")
    bio = get_section_text(soup, "Teacher Bio")

    price = ""
    price_match = re.search(r"£\s?\d+(?:\.\d{2})?", full_text)
    if price_match:
        price = price_match.group(0)

    suitable_for = get_section_text(soup, "Suitable For")

    safeguarding = ""
    if "no safeguarding submitted" in full_text.lower():
        safeguarding = "Not submitted"
    elif "safeguarding" in full_text.lower():
        sg_match = re.search(r"(Safeguarding[^\n]*)", full_text)
        safeguarding = clean(sg_match.group(1)) if sg_match else ""

    image_tag = soup.find("img", src=re.compile(r"/images/listing/"))
    image_url = image_tag["src"] if image_tag else ""

    return {
        "id": entry_id, "title": title, "teacher_name": teacher_name,
        "address": address, "schedule_next_session": next_session,
        "schedule_recurrence": recurrence, "style_tags": style_tags,
        "qualification_note": qualification_note, "description": description,
        "how_to_book": how_to_book, "equipment_needed": equipment,
        "teacher_bio": bio, "price": price, "suitable_for": suitable_for,
        "safeguarding": safeguarding, "image_url": image_url,
        "url": BASE_URL.format(entry_id),
    }


def load_seen_ids(out_path):
    if not os.path.exists(out_path):
        return set()
    with open(out_path, newline="", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        return {int(row["id"]) for row in reader if row.get("id")}


def scrape(start, end, out_path, delay, driver):
    seen = load_seen_ids(out_path)
    file_exists = os.path.exists(out_path)

    with open(out_path, "a", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=FIELDNAMES)
        if not file_exists:
            writer.writeheader()

        found, checked = 0, 0
        for entry_id in range(start, end + 1):
            if entry_id in seen:
                continue
            checked += 1
            url = BASE_URL.format(entry_id)
            try:
                html = load_rendered_html(driver, url)
            except WebDriverException as e:
                print(f"[{entry_id}] browser error: {e}")
                time.sleep(delay)
                continue

            row = parse_entry(entry_id, html)
            if row:
                writer.writerow(row)
                f.flush()
                found += 1
                print(f"[{entry_id}] OK  -> {row['title']} / {row['teacher_name']}")
            else:
                print(f"[{entry_id}] skip (not a valid listing)")

            time.sleep(delay)

    print(f"\nDone. Checked {checked} new IDs, saved {found} listings to {out_path}")


def probe_id_range(sample_ids, driver):
    for i in sample_ids:
        url = BASE_URL.format(i)
        try:
            html = load_rendered_html(driver, url)
            row = parse_entry(i, html)
            print(f"id={i}: {'valid listing -> ' + row['title'] if row else 'invalid/empty'}")
        except WebDriverException as e:
            print(f"id={i}: browser error: {e}")

In [ ]:
driver = make_driver()
try:
    probe_id_range([1520, 1540, 1550, 1560, 1580], driver)
finally:
    driver.quit()

id=1520: invalid/empty
id=1540: invalid/empty
id=1550: invalid/empty
id=1560: invalid/empty
id=1580: invalid/empty


In [ ]:
driver = make_driver()
try:
    probe_id_range([1505, 1510, 1515], driver)
finally:
    driver.quit()

id=1505: valid listing -> Ayurveda for sleep
id=1510: valid listing -> Janet Priestley
id=1515: invalid/empty


In [ ]:
driver = make_driver()
try:
    scrape(start=1, end=1520, out_path="bwy_raw.csv", delay=0.5, driver=driver)
finally:
    driver.quit()

[1] OK  -> Yoga for wellbeing / Sally KENNEDY
[2] OK  -> Yoga for wellbeing / Sally KENNEDY
[3] OK  -> Hatha Yoga Class / Janet LONG
[4] OK  -> Hatha Flow Yoga in Wimbledon / Alexandra REED
[5] OK  -> Gentle Yoga (Chair based class) / Kate HOLLY
[6] OK  -> Hatha Yoga for All / Kate HOLLY
[7] OK  -> Hatha Yoga for All / Kate HOLLY
[8] OK  -> Yoga with Amanda / Amanda BRAKE
[9] OK  -> General yoga / Amanda BRAKE
[10] OK  -> Yoga with Claire / Claire MESSENGER
[11] OK  -> Yoga with Claire / Claire MESSENGER
[12] OK  -> YogaBirth Pregnancy Yoga @Barkantine Practice / Arlene DUNKLEY-WOOD
[13] OK  -> YogaBirth Pregnancy Yoga @Barkantine Practice / Arlene DUNKLEY-WOOD
[14] OK  -> Yoga for over 40s / Arlene DUNKLEY-WOOD
[15] OK  -> Arlene Angela Dunkley-Wood / Arlene DUNKLEY-WOOD
[16] OK  -> Gong & Sound Bath in Wanstead / Arlene DUNKLEY-WOOD
[17] OK  -> Sound Journey / Arlene DUNKLEY-WOOD
[18] OK  -> Sound Journey / Arlene DUNKLEY-WOOD
[19] OK  -> Sound Journey / Arlene DUNKLEY-WOOD
[20] OK  

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────
import time
import json
import random
import requests
import pandas as pd

from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

MODEL = "gemini-3.5-flash-lite"
API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent?key={GEMINI_API_KEY}"

df = pd.read_csv("bwy_raw.csv")
print(f"Loaded {len(df)} rows from bwy_raw.csv\n")

# ── Classification prompt: 3-tier ───────────────────────────────────
SYSTEM_INSTRUCTIONS = """
You are checking yoga teacher listings for pregnancy/postnatal ("prenatal")
teaching qualifications, using the text provided (qualification_note and
teacher_bio combined).

Classify each listing into exactly one of three tiers:

1. "verified" — the text names ONE of these EXACT three qualification
   routes, and nothing else counts:
   - British Wheel of Yoga (BWY) pregnancy yoga module (must be the
     specific pregnancy/postnatal module — general "BWY teacher" or
     "British Wheel of Yoga" certification alone does NOT count, that
     only proves general teaching status, not the pregnancy module.
     A "Level 4 BWY diploma" or similar general advanced teaching
     diploma also does NOT count, no matter what level number is
     attached — it is a general teaching qualification, not a
     pregnancy/postnatal one)
   - Yoga Alliance Professionals recognised pregnancy/postnatal training,
     85 hours or more (the text must state an explicit number of hours,
     e.g. "85-hour", "100 hour" — words like "extensive", "in-depth", or
     "specialist" training do NOT imply the hour count)
   - A Level 3 (or Level 4) pre and postnatal exercise/yoga award
     (e.g. Active IQ, Focus Awards, or an explicitly named Level 3/4
     pre/postnatal diploma)

   CRITICAL — the "Level 3/4" route only applies when the level number
   is part of THE SAME qualification title as the pregnancy/postnatal
   wording — e.g. "Level 3 Pre and Postnatal Yoga Award" is one award
   name. A level number and a pregnancy mention that appear in different,
   unrelated parts of the bio (e.g. "Level 4 BWY diploma" in one sentence
   and "passionate about supporting women in pregnancy" in a separate,
   unrelated sentence) do NOT combine into a verified qualification. Do
   not stitch together fragments from separate sentences or separate
   claims to build evidence — the evidence phrase must be a single,
   continuous statement naming one qualification, and everything in it
   must come from that one statement, not assembled from multiple parts
   of the bio.

   The evidence phrase itself must contain a concrete marker of one of
   these routes: the awarding body name, "module", "Level 3"/"Level 4",
   a specific hour count, or an explicitly named diploma/award title —
   AND that marker must be grammatically part of the qualification name
   itself, not merely present somewhere in the same bio.//
   If the phrase is only a vague self-description of skill or experience
   — e.g. "trained in pregnancy and postnatal yoga", "qualified in
   prenatal yoga", "specialises in postnatal recovery", "experienced in
   teaching pregnant clients" — with NO awarding body, module name, level,
   or hour count attached, it does NOT qualify as verified. Being trained
   is not the same as naming what the training was. Route these to
   "self_claimed" instead.

   Do NOT count as verified: Birthlight, Well Woman Yoga, Active Birth
   Centre training, or any other named program that is not one of the
   three routes above — no matter how specific, dated, or legitimate it
   sounds. These go to "self_claimed" instead, since they are real
   training but do not match the brief's approved routes.

   Do NOT count as verified: a class title or description that simply
   contains words like "Pre & Post Natal Yoga" or "Pregnancy Yoga" with
   no qualification, awarding body, or training named anywhere in the
   text. A class NAME is not evidence of a qualification.

   Do NOT count as verified: a listing whose subject is unrelated to
   yoga teaching (e.g. gong baths, sound healing, reflexology, retreats)
   just because the teacher's general bio elsewhere mentions pregnancy —
   the mention must relate to actual pregnancy/postnatal teaching
   content, not be incidental background in a bio attached to an
   unrelated class.

   If you are uncertain whether a phrase meets this bar, default to
   "self_claimed" rather than "verified".

2. "self_claimed" — the text mentions teaching pregnancy, prenatal,
   postnatal, perinatal, antenatal, "mum and baby", or similar (including
   via a class title alone), but does not meet the strict bar for
   "verified" above. This includes:
   - No qualification named at all, just experience, class titles, or
     vague self-description of being "trained" or "qualified" with no
     named body, module, level, or hour count
   - A named but non-approved specific program (Birthlight, Well Woman
     Yoga, Active Birth Centre, or anything else not in the three
     approved routes)
   - A general BWY or Yoga Alliance certification mentioned (including a
     general Level 3/4 BWY diploma with no pregnancy-specific module
     named), without confirming it was the pregnancy-specific
     module/training
   - A general 200-hour teaching certificate with no perinatal-specific
     training named
   - A level number and a separate, unrelated pregnancy mention
     appearing in the same bio but not part of one qualification title
     (this is self_claimed at most, since the bio shows some general
     credibility plus a pregnancy interest, but does not name a matching
     qualification)

   Before assigning "self_claimed", check polarity and relevance of the
   matched phrase:
   - EXCLUSION/NEGATION: if the phrase says the class is NOT suitable
     for, does not include, or excludes pregnant/postnatal/prenatal
     people (e.g. "not suitable for pregnancy", "please note this class
     is not appropriate during pregnancy"), this is the OPPOSITE of a
     pregnancy-teaching claim. Classify as "unverified", not
     "self_claimed".
   - THIRD-PARTY/UNRELATED CONTEXT: if the pregnancy/postnatal mention
     describes something other than THIS teacher's own teaching or
     training — e.g. a generic mission statement about "supporting women
     through pregnancy and birth" with no class or training named, a
     reference to someone else's pregnancy, or a safety disclaimer telling
     pregnant students to consult a doctor before attending a general
     class — treat it as weak signal only if it is the teacher describing
     their OWN class content or purpose. A safety disclaimer alone
     ("consult your doctor if pregnant") is not a teaching claim; classify
     as "unverified" unless there is separate independent evidence.
   - LISTING RELEVANCE: if the listing itself is for an unrelated
     activity (gong bath, sound healing, reflexology, general retreat)
     and the only pregnancy mention is incidental bio background rather
     than a description of this class's content, classify as
     "unverified" rather than "self_claimed" — an unrelated class page
     is not evidence the teacher offers pregnancy yoga.

3. "unverified" — no mention of pregnancy/prenatal/postnatal/perinatal
   teaching at all in the text, OR the only mention is a negation,
   exclusion, generic safety disclaimer, unrelated context, or incidental
   bio background on an unrelated listing, per the rules above.

For each listing, cite the exact phrase you based the decision on,
copied verbatim as ONE continuous span of the source text — never
combine wording from two different sentences or locations into a single
evidence string (or null for "unverified").

Return ONLY a JSON array, one object per listing, in the same order as
given:
[{"id": <id>, "tier": "verified" | "self_claimed" | "unverified", "evidence": "<quoted phrase or null>"}]
No prose, no markdown fences — just the raw JSON array.
"""

def build_batch_prompt(batch_df):
    listings = [
        {
            "id": int(row["id"]),
            "title": row["title"],
            "text": f"{row['qualification_note']} {row['teacher_bio']}".strip(),
        }
        for _, row in batch_df.iterrows()
    ]
    return SYSTEM_INSTRUCTIONS + "\n\nListings:\n" + json.dumps(listings, ensure_ascii=False)

def call_gemini(prompt, timeout=30):
    payload = {"contents": [{"parts": [{"text": prompt}]}]}
    resp = requests.post(API_URL, json=payload, timeout=timeout)
    resp.raise_for_status()
    data = resp.json()
    return data["candidates"][0]["content"]["parts"][0]["text"]

# ── Batch through the dataset ───────────────────────────────────────
BATCH_SIZE = 15
results = []
failed_batches = []
total_batches = (len(df) + BATCH_SIZE - 1) // BATCH_SIZE
run_start = time.time()

for batch_num, start in enumerate(range(0, len(df), BATCH_SIZE), start=1):
    batch = df.iloc[start:start + BATCH_SIZE]
    prompt = build_batch_prompt(batch)
    batch_start = time.time()

    for attempt in range(5):
        try:
            print(f"→ Sending batch {batch_num}/{total_batches} (rows {start}-{start+len(batch)})...")
            raw_text = call_gemini(prompt, timeout=30)
            text = raw_text.strip().strip("```json").strip("```").strip()
            parsed = json.loads(text)
            results.extend(parsed)

            elapsed = time.time() - batch_start
            print(f"  OK in {elapsed:.1f}s — {len(parsed)} rows returned:")

            for r in parsed:
                row_id = r.get("id")
                tier = r.get("tier")
                evidence = r.get("evidence")
                title_lookup = df.loc[df["id"] == row_id, "title"]
                title = title_lookup.iloc[0] if not title_lookup.empty else "?"
                evidence_display = f'"{evidence}"' if evidence else "—"
                print(f"    id={row_id:<5} tier={tier:<13} title={title[:40]:<40} evidence={evidence_display}")

            break
        except Exception as e:
            wait = (2 ** attempt) + random.uniform(0, 1)
            print(f"  Batch {batch_num}/{total_batches} failed (attempt {attempt+1}/5): {e}")
            print(f"  Retrying in {wait:.1f}s...")
            time.sleep(wait)
    else:
        print(f"Batch {batch_num}/{total_batches} FAILED after all retries — marking as PARSE_FAILED")
        failed_batches.append(start)
        for _, row in batch.iterrows():
            results.append({"id": int(row["id"]), "tier": "PARSE_FAILED", "evidence": None})
            print(f"    id={row['id']:<5} tier=PARSE_FAILED  title={row['title'][:40]}")

    time.sleep(1)

    running_counts = pd.Series([r.get("tier") for r in results]).value_counts().to_dict()
    print(f"  Running total ({len(results)}/{len(df)} rows): {running_counts}\n")

total_elapsed = time.time() - run_start
print(f"{'='*60}")
print(f"Finished in {total_elapsed/60:.1f} minutes")

# ── Merge back onto raw dataframe ───────────────────────────────────
flags_df = pd.DataFrame(results)
merged = df.merge(flags_df, on="id", how="left")

merged.to_csv("bwy_gemini_tiered.csv", index=False)
print(f"Saved bwy_gemini_tiered.csv ({len(merged)} rows)\n")

# ── Final report ─────────────────────────────────────────────────────
print("Final tier breakdown:")
print(merged["tier"].value_counts(dropna=False))

print(f"\nBatches that failed all retries: {failed_batches if failed_batches else 'none'}")

verified_ids = merged.loc[merged["tier"] == "verified", "id"].tolist()
print(f"\nVerified listing IDs ({len(verified_ids)}): {verified_ids}")

print("\nSample of 'verified' evidence:")
print(merged.loc[merged["tier"] == "verified", ["id", "teacher_name", "evidence"]].head(10).to_string(index=False))

print("\nSample of 'self_claimed' evidence:")
print(merged.loc[merged["tier"] == "self_claimed", ["id", "teacher_name", "evidence"]].head(10).to_string(index=False))

Loaded 1511 rows from bwy_raw.csv

→ Sending batch 1/101 (rows 0-15)...
  OK in 1.5s — 15 rows returned:
    id=1     tier=unverified    title=Yoga for wellbeing                       evidence=—
    id=2     tier=unverified    title=Yoga for wellbeing                       evidence=—
    id=3     tier=unverified    title=Hatha Yoga Class                         evidence=—
    id=4     tier=unverified    title=Hatha Flow Yoga in Wimbledon             evidence=—
    id=5     tier=unverified    title=Gentle Yoga (Chair based class)          evidence=—
    id=6     tier=unverified    title=Hatha Yoga for All                       evidence=—
    id=7     tier=unverified    title=Hatha Yoga for All                       evidence=—
    id=8     tier=unverified    title=Yoga with Amanda                         evidence=—
    id=9     tier=unverified    title=General yoga                             evidence=—
    id=10    tier=unverified    title=Yoga with Claire                         eviden

In [1]:
!pip install -q google-generativeai pandas

import pandas as pd
import google.generativeai as genai
import json, re, time

# Put your key in Colab's secret manager (key icon in sidebar) as GEMINI_API_KEY,
# or just paste it here directly.
from google.colab import userdata

GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")  # or: GEMINI_API_KEY = "your-key-here"

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel("gemini-2.5-flash")  # fast + cheap, good enough for this

/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [4]:
from google.colab import files
uploaded = files.upload()  # upload bwy_gemini_tiered__2_.csv when prompted

filename = list(uploaded.keys())[0]
df = pd.read_csv(filename)
print(len(df), "rows")
df.head()
# Keep only verified rows
df = df[df["tier"].str.lower() != "unverified"].reset_index(drop=True)
print(f"{len(df)} rows remain after dropping unverified")

Saving bwy_gemini_tiered (2).csv to bwy_gemini_tiered (2) (1).csv
1511 rows
124 rows remain after dropping unverified


In [5]:
import json, re, time

CHECKPOINT_FILE = "bwy_labels_checkpoint.csv"
BATCH_SIZE = 40
MAX_RETRIES = 5

# Resume support: load any labels already saved from a previous run
try:
    checkpoint = pd.read_csv(CHECKPOINT_FILE)
    labeled_ids = set(checkpoint["id"])
    print(f"Resuming — {len(labeled_ids)} rows already labeled")
except FileNotFoundError:
    checkpoint = pd.DataFrame(columns=["id", "region"])
    labeled_ids = set()

def classify_batch(rows):
    """rows: list of (id, address) tuples. Returns dict {id: region}."""
    numbered = "\n".join(f"{rid}: {addr}" for rid, addr in rows)
    prompt = f"""You are classifying UK addresses by region.
For each address below (given as "id: address"), decide if it is in:
- "London" (Greater London, any London borough/postcode E, EC, N, NW, SE, SW, W, WC)
- "Essex" (Essex county, including RM/IG/CM/CO/SS postcode areas)
- "Other" (anywhere else, or if the address is empty/unclear)

Addresses:
{numbered}

Respond ONLY with a JSON array of objects, one per address, using the SAME id
given above, e.g.:
[{{"id": 12, "region": "London"}}, {{"id": 13, "region": "Essex"}}]
No explanation, no markdown fences. Include every id exactly once."""

    for attempt in range(MAX_RETRIES):
        try:
            response = model.generate_content(prompt)
            text = response.text.strip()
            text = re.sub(r"^```json|```$", "", text, flags=re.MULTILINE).strip()
            parsed = json.loads(text)
            result = {int(item["id"]): item["region"] for item in parsed}
            # fill in any ids the model dropped
            for rid, _ in rows:
                result.setdefault(rid, "Other")
            return result
        except Exception as e:
            wait = 2 ** attempt
            print(f"  attempt {attempt+1} failed ({e}), retrying in {wait}s...")
            time.sleep(wait)

    print("  giving up on this batch, defaulting to Other")
    return {rid: "Other" for rid, _ in rows}


# Only process rows not already labeled
rows_to_do = [(int(r["id"]), r["address"] if pd.notna(r["address"]) else "")
              for _, r in df.iterrows() if r["id"] not in labeled_ids]

print(f"{len(rows_to_do)} rows left to process")

for i in range(0, len(rows_to_do), BATCH_SIZE):
    batch = rows_to_do[i:i+BATCH_SIZE]
    result = classify_batch(batch)

    batch_df = pd.DataFrame(
        [{"id": rid, "region": region} for rid, region in result.items()]
    )
    checkpoint = pd.concat([checkpoint, batch_df], ignore_index=True)
    checkpoint.to_csv(CHECKPOINT_FILE, index=False)  # save after every batch

    print(f"Processed {min(i+BATCH_SIZE, len(rows_to_do))}/{len(rows_to_do)} "
          f"(total labeled: {len(checkpoint)})")
    time.sleep(1)

print("Done.")

124 rows left to process
Processed 40/124 (total labeled: 40)
Processed 80/124 (total labeled: 80)
Processed 120/124 (total labeled: 120)
Processed 124/124 (total labeled: 124)
Done.


In [6]:
checkpoint = pd.read_csv(CHECKPOINT_FILE)
df = df.merge(checkpoint, on="id", how="left")
df["region"] = df["region"].fillna("Other")

print(df["region"].value_counts())
pd.set_option("display.max_colwidth", 100)
df[["id", "title", "address", "region"]]

region
Other     107
London     16
Essex       1
Name: count, dtype: int64


,id,title,address,region
0,12,YogaBirth Pregnancy Yoga @Barkantine Practice,"Barkantine Practice, E14 121, West Ferry Road, London, E14 8JH",London
1,13,YogaBirth Pregnancy Yoga @Barkantine Practice,"121, West Ferry Road, London, E14 8JH",London
2,16,Gong & Sound Bath in Wanstead,NaN,Other
3,17,Sound Journey,NaN,Other
4,18,Sound Journey,NaN,Other
...,...,...,...,...
119,1458,Yoga for Pregnancy,"Manton Village Hall Preshute Lane, Manton, Wiltshire, SN8 4HQ",Other
120,1460,Mokshyoga,"Phillimore Community centre, Phillimore Place, Hertfordshire, WD7 8NN",Other
121,1468,Sally Rainbowchild Pregnancy Yoga,NaN,Other
122,1487,Gong and Sound Bath with Arlene Dunkley-Wood,"Wanstead Quaker Meeting House, Bush Road, E11 3AU",London


In [7]:
df.to_csv("bwy_london_essex_only.csv", index=False)

In [8]:
filtered = df[df["region"] != "Other"].copy()
print(f"Kept {len(filtered)} of {len(df)} rows")
print(filtered["region"].value_counts())

filtered.to_csv("bwy_london_essex_only_final.csv", index=False)

filtered.head()

Kept 17 of 124 rows
region
London    16
Essex      1
Name: count, dtype: int64


,id,title,teacher_name,address,schedule_next_session,schedule_recurrence,style_tags,qualification_note,description,how_to_book,equipment_needed,teacher_bio,price,suitable_for,safeguarding,image_url,url,tier,evidence,region
0,12,YogaBirth Pregnancy Yoga @Barkantine Practice,Arlene DUNKLEY-WOOD,"Barkantine Practice, E14 121, West Ferry Road, London, E14 8JH",NaN,"Occurs , Every Thursday at 13:02",Pregnancy yoga,Entry Description Prenatal Yoga is a crucial practice to adopt from 14 weeks of pregnancy. This ...,Prenatal Yoga is a crucial practice to adopt from 14 weeks of pregnancy.,Equipment Needed,Teacher Bio,"£110.00 Suitable For: Beginners, Intermediate Practitioners, Experienced Practitioners",£110.00,NaN,Not submitted,/images/listing/d2749d76-6cfe-4bff-bf08-d29d909d3617.jpg,https://portal.bwy.org.uk/user/public-entries/12,self_claimed,Prenatal Yoga is a crucial practice to adopt from 14 weeks of pregnancy.,London
1,13,YogaBirth Pregnancy Yoga @Barkantine Practice,Arlene DUNKLEY-WOOD,"121, West Ferry Road, London, E14 8JH",NaN,"Occurs Weekly, Every Thursday at 13:02",Pregnancy yoga,Entry Description Prenatal Yoga is a crucial practice to adopt from 14 weeks of pregnancy. This ...,Prenatal Yoga is a crucial practice to adopt from 14 weeks of pregnancy.,www.arlenedunkleywood.co.uk/bookonline,Yoga mat and cushions provided. Bring your own bottle of water. Leggins and stocking feet.,"£110.00 Suitable For: Beginners, Intermediate Practitioners, Experienced Practitioners",£110.00,NaN,Not submitted,/images/listing/nEAl6ji7fx3UaHfaERgTfTxaf8tXaRP5eYNHWWE7.jpg,https://portal.bwy.org.uk/user/public-entries/13,self_claimed,Prenatal Yoga is a crucial practice to adopt from 14 weeks of pregnancy.,London
5,19,Sound Journey,Arlene DUNKLEY-WOOD,"Good Shepherd Studios 15a, Davies Lane, Leytonstone, E11 3DR",NaN,Occurs Monthly,"Sound bath, Gong bath",Entry Description Sound is a powerful tool to help you relax and release stress. But it is also ...,Sound is a powerful tool to help you relax and release stress. But it is also a way to let go of...,https://www.eventbrite.co.uk/e/sound-journey-good-shepherd-studios-leytonstone-with-arlene-dunkl...,"Yoga mats, cushions, blankets are all provided. Please bring some water to drink at the end. Com...","£25.00 Suitable For: Beginners, Intermediate Practitioners, Experienced Practitioners, Student T...",£25.00,NaN,Not submitted,/images/listing/mn2WSvc5lw5WwtXI92S4cWWrPS1fxfsG9SQepado.jpg,https://portal.bwy.org.uk/user/public-entries/19,self_claimed,"My work is deeply rooted in promoting health and vitality, especially in pregnancy and birth",London
11,53,Sound Journey @ Good Shepherd Studios Leytonstone,Arlene DUNKLEY-WOOD,"Good Shepherd Studios 15a, Davies Lane, Leytonstone, E11 3DR",NaN,Occurs Monthly,Sound bath,Entry Description Sound Bath/Journey are a wonderful way of letting go of stress and finding tha...,Sound Bath/Journey are a wonderful way of letting go of stress and finding that calm you need to...,www.arlenedunkleywood.co.uk/bookonline,"Yoga mat, socks, cushions, blanket, bottle of water.","£25.00 Suitable For: Beginners, Intermediate Practitioners, Experienced Practitioners, Student T...",£25.00,NaN,Not submitted,/images/listing/pEIkk5QJaduavtZoEguM5xtn8HsWcX96WfCDTe8x.png,https://portal.bwy.org.uk/user/public-entries/53,self_claimed,"My work is deeply rooted in promoting health and vitality, especially in pregnancy and birth, an...",London
26,133,Julia Davis 90 minute Yoga Class,Julia DAVIS,"Finchley Yoga 1 Maple Close, Finchley, N3 1AS",03/08/2026,"Occurs Weekly, Every Monday,Thursday at 09:30","Hatha yoga, Meditation, Breathwork, Yoga nidra",Entry Description 90 minute yoga class with Julia Davis - Julia has over 20 years of experience ...,90 minute yoga class with Julia Davis - Julia has over 20 years of experience teaching. The clas...,https://finchleyyoga.com/,Alll equipment provided. Recommend you bring your own mat if you have one.,"£0.00 Suitable For: Beginners, Intermediate Prac

In [9]:
print(filtered.columns.tolist())
filtered[["id", "address", "region"]].head()

['id', 'title', 'teacher_name', 'address', 'schedule_next_session', 'schedule_recurrence', 'style_tags', 'qualification_note', 'description', 'how_to_book', 'equipment_needed', 'teacher_bio', 'price', 'suitable_for', 'safeguarding', 'image_url', 'url', 'tier', 'evidence', 'region']


,id,address,region
0,12,"Barkantine Practice, E14 121, West Ferry Road, London, E14 8JH",London
1,13,"121, West Ferry Road, London, E14 8JH",London
5,19,"Good Shepherd Studios 15a, Davies Lane, Leytonstone, E11 3DR",London
11,53,"Good Shepherd Studios 15a, Davies Lane, Leytonstone, E11 3DR",London
26,133,"Finchley Yoga 1 Maple Close, Finchley, N3 1AS",London


In [10]:
import re
from collections import defaultdict

def get_postcode(addr):
    """Extract postcode as a dedup key for identifying the same physical venue."""
    if not addr or not isinstance(addr, str):
        return None
    m = re.search(r'([A-Za-z]{1,2}\d[\dA-Za-z]?\s*\d[A-Za-z]{2})\s*$', addr.strip())
    return m.group(1).upper().replace(' ', '') if m else addr.strip().lower()

def uniq_join(series, sep=' | '):
    """Join distinct values, preserving every one, no repeats."""
    seen = []
    for v in series:
        v = str(v).strip()
        if v and v not in seen and v != 'nan':
            seen.append(v)
    return sep.join(seen)

def dedup_addresses(addresses):
    """Group addresses sharing a postcode (same venue, different text) —
    keeps every original string, doesn't delete any."""
    by_postcode = defaultdict(list)
    for a in addresses:
        a = str(a).strip()
        if not a or a == 'nan':
            continue
        pc = get_postcode(a)
        if a not in by_postcode[pc]:
            by_postcode[pc].append(a)

    venue_strings = []
    for pc, variants in by_postcode.items():
        venue_strings.append(" / ".join(variants) if len(variants) > 1 else variants[0])
    return "  ;  ".join(venue_strings), len(by_postcode)

rows = []
for teacher, group in filtered.groupby('teacher_name'):
    addr_merged, n_venues = dedup_addresses(group['address'])
    rows.append({
        'teacher_name': teacher,
        'num_listings': len(group),
        'num_unique_venues': n_venues,
        'titles': uniq_join(group['title']),
        'addresses': addr_merged,
        'style_tags': uniq_join(group['style_tags']),
        'price': uniq_join(group['price']),
        'region': uniq_join(group['region']),
        'ids': uniq_join(group['id'].astype(str)),
        'urls': uniq_join(group['url']),
    })

merged = pd.DataFrame(rows)
print(f"{len(merged)} unique teachers")
merged.to_csv('bwy_teachers_merged.csv', index=False)
merged

7 unique teachers


,teacher_name,num_listings,num_unique_venues,titles,addresses,style_tags,price,region,ids,urls
0,Arlene DUNKLEY-WOOD,6,4,YogaBirth Pregnancy Yoga @Barkantine Practice | Sound Journey | Sound Journey @ Good Shepherd St...,"Barkantine Practice, E14 121, West Ferry Road, London, E14 8JH / 121, West Ferry Road, London, E...","Pregnancy yoga | Sound bath, Gong bath | Sound bath | Sound healing",£110.00 | £25.00 | £20.00 | £30.00,London,12 | 13 | 19 | 53 | 188 | 1487,https://portal.bwy.org.uk/user/public-entries/12 | https://portal.bwy.org.uk/user/public-entries...
1,Bernadette KING,1,1,Well Woman Yoga,"The Studio, Blackheath Complementary Health Centre 184/6 Westcombe Hill, SE3 7DH","Relaxation, Yoga for stress, Anatomy & physiology, Asana, Pranayama, Yoga nidra",£0.00,London,630,https://portal.bwy.org.uk/user/public-entries/630
2,Evy DEMETRIOU,3,1,The British Wheel of Yoga | Principles of Practising Yoga through Breath and Understanding the A...,"Private Yoga Studio Silvercliffe Gardens, Hertfordshire, EN4 9QT",Hatha yoga,£12.00,London,916 | 917 | 918,https://portal.bwy.org.uk/user/public-entries/916 | https://portal.bwy.org.uk/user/public-entrie...
3,Joan JAGGERNAUTH,1,1,"Hatha Yoga, can include lower back care. For pregnant ladies, trained support with trained pregn...","St Andrews Methodist Church Hall, Herongate, Brentwood, Essex Billericay Road, Herongate, Brentw...",Hatha yoga,£8.00,Essex,783,https://portal.bwy.org.uk/user/public-entries/783
4,Julia DAVIS,3,1,Julia Davis 90 minute Yoga Class | Circle Holding In Person Training for Yoga Teachers | Yoga wi...,"Finchley Yoga 1 Maple Close, Finchley, N3 1AS","Hatha yoga, Meditation, Breathwork, Yoga nidra | Hatha yoga",£0.00 | £255.00,London,133 | 345 | 347,https://portal.bwy.org.uk/user/public-entries/133 | https://portal.bwy.org.uk/user/public-entrie...
5,Julie KRAUSZ,2,1,Tuesday eve Pregnancy Yoga class in Willesden Green with Julie Krausz | Thursday Postnatal Yoga ...,"ZenW2 Unit 12, Queens Parade,, Walm Lane, NW2 5HT","Pregnancy yoga, Scaravelli yoga | Scaravelli yoga, Postnatal yoga",£22.00,London,887 | 888,https://portal.bwy.org.uk/user/public-entries/887 | https://portal.bwy.org.uk/user/public-entrie...
6,Sunnah ROSE,1,1,Beginners Yoga in East Finchley,"Church Lane, Church Lane, East Finchley, N2 0TH",Hatha yoga,£15.00,London,1281,https://portal.bwy.org.uk/user/public-entries/1281


In [3]:
"""
One row per teacher. venue_addresses, postcodes, latitudes, longitudes
are each pipe-separated (|) and aligned by position (venue 1's postcode/
lat/lon are the 1st items in each list, etc).

Run in Colab (needs internet access to postcodes.io).

Input:  bwy_teachers_enriched (2).csv
Output: bwy_teachers_geocoded_per_teacher.csv
"""
import pandas as pd
import requests
import re

INPUT_CSV = "/content/bwy_teachers_enriched (2).csv"
OUTPUT_CSV = "bwy_teachers_geocoded_per_teacher.csv"
BULK_URL = "https://api.postcodes.io/postcodes"

POSTCODE_RE = re.compile(
    r'\b([A-PR-UWYZ][A-HK-Y]?\d[A-Z\d]?\s*\d[ABD-HJLNP-UW-Z]{2})\b',
    re.IGNORECASE
)

df = pd.read_csv(INPUT_CSV)

# --- Step 1: extract venues + postcodes per teacher ---
teacher_rows = []
all_postcodes = set()
for _, row in df.iterrows():
    addresses_field = row.get("addresses")
    venues = [v.strip() for v in re.split(r'\s*;\s*', str(addresses_field))] if pd.notna(addresses_field) else []
    postcodes = []
    for venue in venues:
        matches = POSTCODE_RE.findall(venue)
        norm = []
        for m in matches:
            m2 = m.upper().replace(' ', '')
            m2 = m2[:-3] + ' ' + m2[-3:]
            if m2 not in norm:
                norm.append(m2)
        pc = norm[0] if norm else ""
        postcodes.append(pc)
        if pc:
            all_postcodes.add(pc)
    teacher_rows.append({
        "teacher_name": row["teacher_name"],
        "venue_addresses": venues,
        "postcodes": postcodes,
    })

# --- Step 2: geocode all unique postcodes in bulk ---
lookup = {}
pc_list = list(all_postcodes)
for i in range(0, len(pc_list), 100):
    batch = pc_list[i:i+100]
    resp = requests.post(BULK_URL, json={"postcodes": batch}, timeout=15)
    resp.raise_for_status()
    for item in resp.json()["result"]:
        pc = item["query"]
        result = item["result"]
        lookup[pc] = {
            "latitude": result["latitude"] if result else None,
            "longitude": result["longitude"] if result else None,
        }

# --- Step 3: build final per-teacher rows with aligned pipe-separated fields ---
final_rows = []
for t in teacher_rows:
    lats = [str(lookup.get(pc, {}).get("latitude", "")) if pc else "" for pc in t["postcodes"]]
    lons = [str(lookup.get(pc, {}).get("longitude", "")) if pc else "" for pc in t["postcodes"]]
    final_rows.append({
        "teacher_name": t["teacher_name"],
        "venue_addresses": " | ".join(t["venue_addresses"]),
        "postcodes": " | ".join(t["postcodes"]),
        "latitude": " | ".join(lats),
        "longitude": " | ".join(lons),
    })

out = pd.DataFrame(final_rows)
out.to_csv(OUTPUT_CSV, index=False)
print(out.to_string(index=False))

       teacher_name                                                                                                                                                                                                                                                                                    venue_addresses                             postcodes                                      latitude                                  longitude
Arlene DUNKLEY-WOOD Barkantine Practice, E14 121, West Ferry Road, London, E14 8JH / 121, West Ferry Road, London, E14 8JH | Good Shepherd Studios 15a, Davies Lane, Leytonstone, E11 3DR | Leytonestone United Free Church 55 Wallwood Road, Leytonstone, E11 1AY | Wanstead Quaker Meeting House, Bush Road, E11 3AU E14 8JH | E11 3DR | E11 1AY | E11 3AU 51.500807 | 51.564475 | 51.570852 | 51.569488 -0.025698 | 0.011904 | 0.003469 | 0.019528
    Bernadette KING                                                                                                         